In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 9.7 MB/s eta 0:00:02
     ------------- -------------------------- 4.2/12.8 MB 15.9 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 16.4 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 18.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 17.8 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
!python ../preprocessing.py

Loading data...
Data quality check:
Shape: (27131, 12)
Total Papers: 27131
Total Columns: 12

Missing values per column:
id                 0
submitter          0
authors            0
title              0
comments       10158
journal-ref    24190
doi            24643
abstract           0
report-no      25780
categories         0
versions           0
year               0
dtype: int64

Duplicate IDs: 0

Duplicate Abstracts: 5

Empty Abstracts: 0

Short Abstracts (<10 words): 5

Year Range: 2007 to 2021
Papers per Year:
year
2007      87
2008     131
2009     197
2010     310
2011     547
2012     600
2013    1455
2014     854
2015    1014
2016    1622
2017    2521
2018    2844
2019    2775
2020    4344
2021    7830
Name: count, dtype: int64

Cleaning data...
After dropping missing abstracts: 27131 papers. So removed 0
After dropping duplicate IDs: 27131 papers. So removed 0
After dropping duplicate abstracts: 27126 papers. So removed 5
After dropping empty abstracts: 27126 papers. So rem

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\11873\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!

100%|██████████| 27121/27121 [01:20<00:00, 335.91it/s]


In [5]:
import sys
# I will add the upper level folder (..) to this search list
sys.path.append('..')

In [6]:
import re
import numpy as np
import pandas as pd

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, TruncatedSVD, LatentDirichletAllocation
from sklearn.preprocessing import normalize

In [9]:
from tqdm import tqdm

In [10]:
df = pd.read_pickle("data_preprocessed.pkl")

In [15]:
print(f"Total number of filtered AI/ML/NLP papers: {len(df)}")

Total number of filtered AI/ML/NLP papers: 27121


In [16]:
display(df.head(3))

,id,submitter,authors,title,comments,journal-ref,doi,abstract,report-no,categories,versions,year,cleaned_abstract,period
0,0704.1274,Dev Rajnarayan,David H. Wolpert and Dev G. Rajnarayan,Parametric Learning and Monte Carlo Optimization,None,None,None,This paper uncovers and explores the close r...,None,[cs.LG],[v1],2007,paper uncovers explores close relationship mon...,2005-2009
1,0704.1394,Tarik Had\v{z}i\'c,"Tarik Hadzic, Rune Moller Jensen, Henrik Reif ...",Calculating Valid Domains for BDD-Based Intera...,None,None,None,In these notes we formally describe the func...,None,[cs.AI],[v1],2007,notes formally describe functionality calculat...,2005-2009
2,0704.2010,Juliana Bernardes,"Juliana S Bernardes, Alberto Davila, Vitor San...",A study of structural properties on profiles HMMs,"6 pages, 7 figures",None,None,Motivation: Profile hidden Markov Models (pH...,None,[cs.AI],"[v1, v2]",2007,motivation profile hidden markov models phmms ...,2005-2009


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
# Now I will select the already cleaned summary column and will convert it into a list format to use in the TF-IDF model
abstracts = df['cleaned_abstract'].dropna().tolist()
# Unigrams
# I will remove the excessively common words that appear in more than 90% of articles
# Also, I will remove the rare words and typos that appear less than 5 times in 30,000 articles
# Then I will check the overall weight-to-word frequency ratio and will extract only the highest 15 features, in order to see the effect
tfidf_uni = TfidfVectorizer(stop_words='english', ngram_range=(1, 1), max_df=0.9, min_df=5, max_features=15)
tfidf_uni.fit(abstracts)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [19]:
print("The top 15 Unigrams with the highest frequency and weight:")
#  Here, I am displaying the results, which I selected from the 5,000 most important words
print(tfidf_uni.get_feature_names_out())

The top 15 Unigrams with the highest frequency and weight:
['approach' 'based' 'data' 'language' 'learning' 'model' 'models' 'neural'
 'paper' 'performance' 'propose' 'results' 'task' 'training' 'using']


In [21]:
# Bigrams
# I will remove the excessively common words that appear in more than 90% of articles
# Also, I will remove the rare words and typos that appear less than 5 times in 30,000 articles
# Then I will check the overall weight-to-word frequency ratio and will extract only the highest 15 features, in order to see the effect

tfidf_bi = TfidfVectorizer(stop_words='english', ngram_range=(2, 2), max_df=0.9, min_df=5, max_features=15)
tfidf_bi.fit(abstracts)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(2, ...)"


In [22]:
print("The top 15 Bigrams with the highest frequency and weight")
print(tfidf_bi.get_feature_names_out())

The top 15 Bigrams with the highest frequency and weight
['deep learning' 'experimental results' 'language models'
 'language processing' 'machine learning' 'machine translation'
 'natural language' 'neural network' 'neural networks' 'paper propose'
 'pre trained' 'real world' 'reinforcement learning' 'state art'
 'training data']


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# From the above results, I will make a note of all the common words, such as “paper” and “result”
# Mainly because they do not help to distinguish between research topics
academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art' 
}

# Then, I will merge my custom vocabulary list with inbuilt English stopwords list
# This will basically help me to filter out the basic vocabulary
my_stop_words = list(ENGLISH_STOP_WORDS.union(academic_boilerplate))

# Now, I will run the upgraded bigram extractor
tfidf_bi_clean = TfidfVectorizer(
    stop_words=my_stop_words, 
    ngram_range=(2, 2), 
    max_df=0.9, 
    min_df=5, 
    max_features=15
)
tfidf_bi_clean.fit(abstracts)

print("15 Bigrams after cutting out the academic nonsense:")
# I will display the important words from the results
print(tfidf_bi_clean.get_feature_names_out())

15 Bigrams after cutting out the academic nonsense:
['deep learning' 'end end' 'language models' 'language processing'
 'large scale' 'machine learning' 'machine translation' 'natural language'
 'neural network' 'neural networks' 'pre trained' 'real world'
 'reinforcement learning' 'training data' 'word embeddings']


In [24]:
# Trigrams
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art' 
}

my_stop_words = list(ENGLISH_STOP_WORDS.union(academic_boilerplate))

# Now, I will run the upgraded trigram extractor
tfidf_tri = TfidfVectorizer(
    stop_words=my_stop_words,
    ngram_range=(3,3),
    max_df=0.9,
    min_df=5,
    max_features=15)

tfidf_tri.fit(abstracts)

print("The top 15 Trigrams with the highest frequency and weight")
print(tfidf_tri.get_feature_names_out())

The top 15 Trigrams with the highest frequency and weight
['deep learning models' 'deep neural networks' 'language processing nlp'
 'long short term' 'machine learning models' 'machine translation nmt'
 'named entity recognition' 'natural language processing'
 'natural language understanding' 'neural machine translation'
 'pre trained language' 'recurrent neural network'
 'recurrent neural networks' 'short term memory' 'trained language models']


In [25]:
academic_boilerplate = {
    'paper', 'propose', 'proposed', 'approach', 'method', 'methods', 
    'results', 'experimental', 'experiment', 'show', 'shows', 
    'using', 'based', 'performance', 'task', 'state', 'art'
}

# Here, I will configure the final TF-IDF vectorizer
tfidf_final = TfidfVectorizer(
    stop_words=list(academic_boilerplate), 
    ngram_range=(2,2),
    max_df=0.9,
    min_df=5,
    max_features=5000 # Keeping only the most important 5,000 features
)

X_tfidf = tfidf_final.fit_transform(abstracts)


In [26]:
print(f"The shape of the matrix: {X_tfidf.shape}")
print("We have 30529 papers and extract the 5000 most core two-word features for them")

The shape of the matrix: (27121, 5000)
We have 30529 papers and extract the 5000 most core two-word features for them


In [27]:
feature_names_final = tfidf_final.get_feature_names_out()